# 4 · Fraud detection on 590K real transactions

**Data:** [IEEE-CIS Fraud Detection](https://www.kaggle.com/competitions/ieee-fraud-detection) (Vesta Corporation):
590,540 e-commerce card transactions, 3.5% fraud, 434 columns (many anonymised). These are card-not-present
payments, not UPI, but the problem is the same: rare, adaptive fraud hidden among legitimate digital payments.

**Setup:** put your Kaggle API token in `~/.kaggle/kaggle.json`, accept the competition rules, then run `make data`.

**Design decisions**
1. **Time-based split** (train on the past, test on the future). A random split leaks future card behaviour.
2. **PR-AUC** as the headline metric, because accuracy and ROC-AUC flatter models under 3.5% prevalence.
3. Imbalance handled by **class weights vs undersampling**, compared on validation. No SMOTE.
4. **Threshold chosen by business cost** on validation, then applied once to test.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 160)
from upi import config, db, plots
from upi.fraud import data, model, pipeline
from upi.fraud.features import FraudFeatureBuilder

In [ ]:
df = data.load()          # first run parses the CSVs and caches a parquet file
print(f"{len(df):,} rows x {df.shape[1]} columns | fraud rate {df.isFraud.mean():.2%}")

## EDA

In [ ]:
missing = df.isna().mean().sort_values(ascending=False)
print(f"columns >50% missing: {(missing > 0.5).sum()} of {df.shape[1]}")
df["TransactionAmt"].describe().round(2)

In [ ]:
plots.fraud_rate_by(df, "ProductCD", "Fraud rate by product code");

In [ ]:
plots.fraud_rate_by(df, "card4", "Fraud rate by card network");

In [ ]:
hours = df[["isFraud"]].assign(hour=(df.TransactionDT // 3600) % 24)
plots.fraud_rate_by(hours, "hour", "Fraud rate by hour of day");

## Train, compare, threshold, test

`train_and_evaluate` runs the whole sequence: time split → feature fit on train → logistic baseline →
3 XGBoost imbalance strategies (early stopping on validation PR-AUC) → cost-optimal threshold → one test evaluation.

In [ ]:
res = pipeline.train_and_evaluate(df)
res["split_info"]

In [ ]:
res['comparison']

In [ ]:
pd.DataFrame({"logistic @0.5": res["baseline_test_metrics"], "xgb @0.5": res["test_metrics_at_0_5"],
              f"xgb @{res['threshold']:.2f} (cost-optimal)": res["test_metrics"]}).round(4)

In [ ]:
plots.pr_curves(res["pr_curves"], res["base_rate_test"]);

In [ ]:
plots.threshold_cost(res["threshold_costs"], res["threshold"]);

In [ ]:
pd.Series(res["business"]).round(3)

## Explainability

In [ ]:
import shap
shap.summary_plot(res["shap_values"], res["shap_sample"], max_display=20)

## Fraud prevention vs user experience

Every false positive is a genuine customer whose payment failed. The cost-optimal threshold makes that trade-off
explicit; in production use three bands: **approve / step-up authentication (OTP) / block**.